# Treatment from the Vreeland IMF data

A country is treated in year *t* if it had at least one arrangement of the relevant type running. Three definitions: narrow (SAF/ESAF/PRGF/ECF), narrow plus EFF, and all arrangements.

**Expected:** 300 country-years narrow, 301 with EFF, 321 broad. 17 groups with one treated side after excluding South Africa.

In [1]:
import sys
sys.path.append("../src")

import pandas as pd

import config
import treatment

In [2]:
units = pd.read_csv(config.WORK / 'units.csv')
skeleton, group_ids = treatment.run(units)

C:\Users\User\Documents\Uni\BA\repo\ba-sap-border-did-main\notebooks\../src\treatment.py:7: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding of latin-1 is being used.  This can happen when a file
has been incorrectly encoded by Stata or some other software. You should verify
the string values returned are correct.
  v = pd.read_stata(config.VREELAND_DTA)


country-years by definition: {'narrow': 300, 'eff': 301, 'broad': 321}
skeleton: 3366 rows, 51 groups
[narrow] groups with one treated and one control side: 19
dropped for South Africa: [np.int64(154), np.int64(163)] -> 17 groups left


In [3]:
print(group_ids)

[124, 134, 173, 224, 321, 503, 532, 574, 685, 789, 829, 1023, 1095, 1101, 1179, 1192, 1243]


In [4]:
ever = skeleton.groupby('FIPS_CNTRY')['narrow'].max()
print('treated:', sorted(ever[ever == 1].index))
print('never treated:', sorted(ever[ever == 0].index))

treated: ['CD', 'CF', 'CG', 'CM', 'CT', 'DJ', 'EK', 'ET', 'GA', 'GH', 'GV', 'IV', 'KE', 'LI', 'LT', 'ML', 'MZ', 'PU', 'SG', 'SL', 'TO', 'TZ', 'UG', 'UV', 'ZA', 'ZI']
never treated: ['AO', 'NI', 'SF', 'SO', 'SU', 'WA', 'WZ']


In [5]:
v = pd.read_stata(config.VREELAND_DTA)
print(v.columns.tolist())
txt = [c for c in v.columns if v[c].dtype == object]
for c in txt:
    hits = sorted(x for x in v[c].dropna().unique() if "sud" in str(x).lower())
    if hits:
        print(c, hits)
print("max year:", v[[c for c in v.columns if "year" in c.lower()]].max().to_dict())

['year', 'ccode_cow', 'ccode_gw', 'ccode_iso3', 'cname_cow', 'cname_imf', 'cname_gw', 'microstate', 'country_syear', 'country_smonth', 'country_sday', 'country_eyear', 'country_emonth', 'country_eday', 'agree_count', 'agree_id', 'agree_smonth', 'agree_sday', 'agree_syear', 'agree_emonth', 'agree_eday', 'agree_eyear', 'stateabb', 'totalamountagreed', 'undrawnbalance', 'type']
ccode_iso3 ['SUD']
cname_cow ['South Sudan', 'Sudan']
cname_imf ['South Sudan', 'Sudan']
cname_gw ['South Sudan', 'Sudan']
stateabb ['SUD']
max year: {'year': 2083.0, 'country_syear': 2011.0, 'country_eyear': 9999.0, 'agree_syear': 2017.0, 'agree_eyear': 2020.0}


C:\Users\User\AppData\Local\Temp\ipykernel_24604\3719227146.py:1: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding of latin-1 is being used.  This can happen when a file
has been incorrectly encoded by Stata or some other software. You should verify
the string values returned are correct.
  v = pd.read_stata(config.VREELAND_DTA)


In [6]:
v = pd.read_stata(config.VREELAND_DTA)
v["year"] = v["year"].astype(int)

print("rows with year > 2024:", int((v["year"] > 2024).sum()))
print(v.loc[v["year"] > 2024,
            ["year", "cname_cow", "type", "agree_syear", "agree_eyear"]]
      .head(15).to_string(index=False))
print()

units = pd.read_csv(config.WORK / "units.csv")
sub = v[v["ccode_cow"].isin(units["COW"].unique()) & v["type"].notna()]
print("last arrangement-year per country in the sample:")
print(sub.groupby("cname_cow")["year"].max().to_string())
print()

print("arrangement types by year from 2012 on:")
print(sub[sub["year"].between(2012, 2024)]
      .groupby(["cname_cow", "year"])["type"]
      .apply(lambda s: ",".join(sorted(set(s)))).unstack(fill_value="-").to_string())
print()

ss = v[(v["cname_cow"] == "South Sudan") & v["type"].notna()]
print(f"South Sudan arrangement-years: {len(ss)}, "
      f"COW {sorted(v.loc[v['cname_cow'] == 'South Sudan', 'ccode_cow'].unique())}")
if len(ss):
    print(ss[["year", "type", "agree_syear", "agree_eyear"]].to_string(index=False))

rows with year > 2024: 59
 year cname_cow type  agree_syear  agree_eyear
 2025     Syria  NaN          NaN          NaN
 2026     Syria  NaN          NaN          NaN
 2027     Syria  NaN          NaN          NaN
 2028     Syria  NaN          NaN          NaN
 2029     Syria  NaN          NaN          NaN
 2030     Syria  NaN          NaN          NaN
 2031     Syria  NaN          NaN          NaN
 2032     Syria  NaN          NaN          NaN
 2033     Syria  NaN          NaN          NaN
 2034     Syria  NaN          NaN          NaN
 2035     Syria  NaN          NaN          NaN
 2036     Syria  NaN          NaN          NaN
 2037     Syria  NaN          NaN          NaN
 2038     Syria  NaN          NaN          NaN
 2039     Syria  NaN          NaN          NaN

last arrangement-year per country in the sample:
cname_cow
Angola                              2011
Burkina Faso                        2017
Cameroon                            2008
Central African Republic            201

C:\Users\User\AppData\Local\Temp\ipykernel_24604\4138986575.py:1: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding of latin-1 is being used.  This can happen when a file
has been incorrectly encoded by Stata or some other software. You should verify
the string values returned are correct.
  v = pd.read_stata(config.VREELAND_DTA)
